In [9]:
import json
import pandas as pd
from collections import defaultdict

In [10]:
#File paths
events_path = "events_England.json"
matches_path = "matches_England.json"
players_path = "../Data/players.json"
teams_path = "../Data/teams.json"

In [11]:
#Load JSON files
with open(events_path, "r", encoding="utf-8") as f:
    events = json.load(f)
with open(matches_path, "r", encoding="utf-8") as f:
    matches = json.load(f)
with open(players_path, "r", encoding="utf-8") as f:
    players = json.load(f)
with open(teams_path, "r", encoding="utf-8") as f:
    teams = json.load(f)

In [12]:
#Create player and team name dictionaries
player_name = {p['wyId']: f"{p['firstName']} {p['lastName']}" for p in players}
team_name = {t['wyId']: t['name'] for t in teams}

In [13]:
#Create list of substitute players for each match
match_subs = {}
for match in matches:
    match_id = match['wyId']
    subs = set()
    for team_data in match.get('teamsData', {}).values():
        for sub in team_data.get('formation', {}).get('substitutions', []):
            if isinstance(sub, dict) and 'playerIn' in sub:
                subs.add(sub['playerIn'])
    match_subs[match_id] = subs

In [14]:
#Count total goals and substitute goals
summary = defaultdict(lambda: {'goals': 0, 'sub_goals': 0, 'sub_scorers': [], 'team_id': None})

for e in events:
    if e['eventName'] == 'Shot' and any(tag.get('id') == 101 for tag in e.get('tags', [])):
        match_id = e['matchId']
        player_id = e['playerId']
        team_id = e['teamId']
        summary[match_id]['goals'] += 1
        summary[match_id]['team_id'] = team_id
        if player_id in match_subs.get(match_id, set()):
            summary[match_id]['sub_goals'] += 1
            summary[match_id]['sub_scorers'].append(player_name.get(player_id, str(player_id)))

In [15]:
# Build final dataframe
df = pd.DataFrame([
    {
        "matchId": mid,
        "team": team_name.get(data["team_id"], "Unknown"),
        "total_goals": data["goals"],
        "sub_goals": data["sub_goals"],
        "sub_scorers": ", ".join(data["sub_scorers"]),
        "rule_triggered": data["sub_goals"] >= 1
    }
    for mid, data in summary.items()
])

In [16]:
print("Number of matches where the rule was triggered:")
print(df["rule_triggered"].value_counts())
print("Examples of matches where the rule was triggered:")
print(df[df["rule_triggered"] == True].head())

Number of matches where the rule was triggered:
rule_triggered
False    257
True      85
Name: count, dtype: int64
Examples of matches where the rule was triggered:
    matchId                  team  total_goals  sub_goals  \
0   2499719               Arsenal            7          2   
2   2499721               Chelsea            5          1   
5   2499724     Manchester United            4          1   
9   2499729               Watford            2          1   
10  2499730  West Bromwich Albion            1          1   

                             sub_scorers  rule_triggered  
0           Aaron Ramsey, Olivier Giroud            True  
2   \u00c1lvaro Borja Morata Mart\u00edn            True  
5                        Anthony Martial            True  
9                         Etienne Capoue            True  
10                       Hal Robson-Kanu            True  
